# Purpose
Simulate baseline genome using msprime and extract ground-truth IBD segments.

# Output
- Tree sequence file
- VCF for IBDSeq input
- CVS file for ground-truth IBD segments

In [2]:
%%time

import msprime
pop_size = 3e5
seq_length = 5e6

ts = msprime.sim_ancestry(
    samples = 200,
    population_size = pop_size,
    sequence_length = seq_length,
    recombination_rate = 1e-8,
    random_seed = 1234)
ts = (msprime.sim_mutations(ts, rate = 1.5e-8, random_seed = 4321))
print(ts)

╔═══════════════════════════╗
║TreeSequence               ║
╠═══════════════╤═══════════╣
║Trees          │    340,935║
╟───────────────┼───────────╢
║Sequence Length│  5,000,000║
╟───────────────┼───────────╢
║Time Units     │generations║
╟───────────────┼───────────╢
║Sample Nodes   │        400║
╟───────────────┼───────────╢
║Total Size     │   88.7 MiB║
╚═══════════════╧═══════════╝
╔═══════════╤═════════╤═════════╤════════════╗
║Table      │Rows     │Size     │Has Metadata║
╠═══════════╪═════════╪═════════╪════════════╣
║Edges      │1,281,741│ 39.1 MiB│          No║
╟───────────┼─────────┼─────────┼────────────╢
║Individuals│      200│  5.5 KiB│          No║
╟───────────┼─────────┼─────────┼────────────╢
║Migrations │        0│  8 Bytes│          No║
╟───────────┼─────────┼─────────┼────────────╢
║Mutations  │  590,099│ 20.8 MiB│          No║
╟───────────┼─────────┼─────────┼────────────╢
║Nodes      │  216,212│  5.8 MiB│          No║
╟───────────┼─────────┼─────────┼────────────╢

In [8]:
ts.dump("simulated_data.trees")

In [9]:
import numpy as np

print("Trees:", ts.num_trees)
print("Sites:", ts.num_sites)
print(f"Diversity: {ts.diversity():.4f}")


# Total segregating sites
all_positions = np.unique([int(site.position) for site in ts.sites()])
all_spacings = np.diff(all_positions)

S = len(ts.sites())
mean_all_spacing = np.mean(all_spacings)
sd_all_spacing = np.std(all_spacings)
print(f"Total Segregating Sites (S): {S}")
print(f"Mean Spacing across pool: {mean_all_spacing:.2f} bp")
print(f"SD of Spacing across pool: {sd_all_spacing:.2f} bp\n")

# Mean SNP spacing across pool
genotypes = ts.genotype_matrix()
num_haplotypes = genotypes.shape[1]

pairwise_means = []
pairs_count = 0
for i in range(0, min(num_haplotypes, 40), 2):
    diff_indices = np.where(genotypes[:, i] != genotypes[:, i+1])[0]
    
    if len(diff_indices) > 1:
        pair_positions = np.unique([int(ts.site(idx).position) for idx in diff_indices])
        pair_spacings = np.diff(pair_positions)
        
        
        pairwise_means.append(np.mean(pair_spacings))


mean_pairwise_spacing = np.mean(pairwise_means)
sd_pairwise_spacing = np.std(pairwise_means)
print("--- (Pairwise SNPs) ---")
print(f"Nucleotide Diversity (pi): {ts.diversity():.4f}")
print(f"Mean Pairwise SNP Spacing: {mean_pairwise_spacing:.2f} bp")
print(f"SD of Pairwise SNP Spacing: {sd_pairwise_spacing:.2f} bp")

Diversity: 0.0178
Total Segregating Sites (S): 555402
Mean Spacing across pool: 9.00 bp
SD of Spacing across pool: 8.77 bp

--- (Pairwise SNPs) ---
Nucleotide Diversity (pi): 0.0178
Mean Pairwise SNP Spacing: 56.59 bp
SD of Pairwise SNP Spacing: 1.53 bp


In [4]:
%%time

ts = ts.simplify()
ibd = ts.ibd_segments(
    store_pairs=True,
    store_segments=True,
    min_span = 2e4
) 

import pandas as pd
ibd_truth = []

for pair in ibd:
    node_i, node_j = pair 
    
    i, j = (node_i // 2, node_j // 2) 
 
    for seg in ibd[pair]:
        ibd_truth.append((i,j,seg.left,seg.right))

df = pd.DataFrame(ibd_truth, columns=["id1", "id2", "start", "end"])
df["length"] = df["end"] - df["start"]                     

df.to_csv("..data/ibd_Truth.csv", index=False)


CPU times: user 669 ms, sys: 41.2 ms, total: 710 ms
Wall time: 714 ms


In [34]:
import sys
small_ts = ts.keep_intervals([[0, 5e6]])
small_ts = small_ts.trim()
with open("data../An_Simulated_.vcf", "w") as f:
    small_ts.write_vcf(f)
